In [ ]:
import os
import re
import time
import torch
import utils
import pickle
import numpy as np

from torch import nn
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset

# PARAMETRI

In [ ]:
EPOCHS = 15
BATCH_SIZE = 128
LEARNING_RATE = 0.001
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')

print(f"DEVICE: {DEVICE}")

# DATASET

In [ ]:
from datasets.datasets import load_ITA_ENG

# DATASET: coppie di frasi inglese-italiano del progetto Tatoeba (www.manythings.org/anki, licenza CC-BY)
df = load_ITA_ENG()
print(f"COPPIE: {len(df)}")
print(df.sample(5, random_state=0).to_string(), "\n")

MIN_LEN = 5
MAX_LEN = 10          # si tengono le coppie con al più MAX_LEN token in entrambe le lingue
NUM_PAIRS = 100000    # coppie usate, estratte a caso
MIN_COUNT = 2         # nel dizionario solo le parole che compaiono almeno MIN_COUNT volte nel training
TEST_SPLIT = 0.1

# TOKENIZZAZIONE: minuscolo, parole e punteggiatura. L'apostrofo resta attaccato alla parola a cui si appoggia:
# in italiano "l'ho" -> "l'" "ho", in inglese "don't" -> "don" "'t"
TOKEN_PATTERNS = {'it': r"\w+'|\w+|[^\w\s]", 'en': r"'\w+|\w+|[^\w\s]"}

def tokenize(sentence, lang):
    return re.findall(TOKEN_PATTERNS[lang], sentence.lower().replace('\u2019', "'"))

# Traduzione dall'italiano (sorgente) all'inglese (destinazione)
pairs = [(tokenize(it, 'it'), tokenize(en, 'en'), en) for it, en in zip(df['italian'], df['english'])]
pairs = [p for p in pairs if MIN_LEN <= len(p[0]) <= MAX_LEN and MIN_LEN <= len(p[1]) <= MAX_LEN]
print(f"COPPIE CON AL PIÙ {MAX_LEN} TOKEN: {len(pairs)}")
rng = np.random.default_rng(0)
pairs = [pairs[i] for i in rng.permutation(len(pairs))[:NUM_PAIRS]]

# Split per frase inglese: le diverse traduzioni della stessa frase ("Sono stanco.", "Sono stanca." -> "I'm tired.")
# vanno tutte nel training o tutte nel test, altrimenti il test conterrebbe frasi quasi già viste
english = sorted({p[2] for p in pairs})
rng.shuffle(english)
test_english = set(english[:int(len(english) * TEST_SPLIT)])
train_pairs = [p for p in pairs if p[2] not in test_english]
test_pairs = [p for p in pairs if p[2] in test_english]

# DIZIONARI, costruiti sul solo training. I primi 4 indici sono riservati ai token speciali
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']   # riempimento, inizio frase, fine frase, parola sconosciuta
PAD, SOS, EOS, UNK = range(len(SPECIALS))

def build_vocab(sentences):
    counts = Counter(t for s in sentences for t in s)
    itos = SPECIALS + [w for w, c in counts.most_common() if c >= MIN_COUNT]
    return itos, {w: i for i, w in enumerate(itos)}

src_itos, src_stoi = build_vocab(p[0] for p in train_pairs)
tgt_itos, tgt_stoi = build_vocab(p[1] for p in train_pairs)
print(f"COPPIE: {len(train_pairs)} di training, {len(test_pairs)} di test")
print(f"DIZIONARI: {len(src_itos)} parole italiane, {len(tgt_itos)} inglesi\n")

def encode(tokens, stoi, length, pad_left):
    ids = [stoi.get(t, UNK) for t in tokens]
    padding = [PAD] * (length - len(ids))
    return padding + ids if pad_left else ids + padding

def make_dataset(pairs):
    # encoder: frase italiana con il padding all'inizio, così la lettura finisce sempre su una parola vera
    src = [encode(p[0], src_stoi, MAX_LEN, pad_left=True) for p in pairs]
    # decoder: legge <sos> + frase inglese e a ogni passo deve predire la parola successiva, fino a <eos>
    tgt_in = [encode(['<sos>'] + p[1], tgt_stoi, MAX_LEN + 1, pad_left=False) for p in pairs]
    tgt_out = [encode(p[1] + ['<eos>'], tgt_stoi, MAX_LEN + 1, pad_left=False) for p in pairs]
    return TensorDataset(torch.tensor(src), torch.tensor(tgt_in), torch.tensor(tgt_out))

train_dataset = make_dataset(train_pairs)
test_dataset = make_dataset(test_pairs)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

src, tgt_in, tgt_out = train_dataset[0]
print("ESEMPIO")
print(f"  encoder legge:   {[src_itos[i] for i in src]}")
print(f"  decoder legge:   {[tgt_itos[i] for i in tgt_in]}")
print(f"  decoder predice: {[tgt_itos[i] for i in tgt_out]}")

# MODELLO

In [ ]:
#Parametri del modello
EMBEDDING_DIM = 256
HIDDEN_SIZE = 256

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.rnn = nn.GRU(embedding_dim, hidden_size, batch_first=True)

    def forward(self, src):                              # (B, MAX_LEN) frase italiana
        _, h = self.rnn(self.embedding(src))             # h: (1, B, HIDDEN_SIZE), lo stato dopo l'ultima parola
        return h                                         # il "vettore di contesto": tutta la frase in HIDDEN_SIZE numeri


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD)
        self.rnn = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, tgt_in, h):                        # (B, T) parole inglesi, (1, B, HIDDEN_SIZE) stato iniziale
        out, h = self.rnn(self.embedding(tgt_in), h)     # (B, T, HIDDEN_SIZE)
        return self.fc(out), h                           # logit della parola successiva a ogni passo (B, T, V), stato finale


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt_in):
        h = self.encoder(src)                            # l'encoder legge la frase italiana...
        logits, _ = self.decoder(tgt_in, h)              # ...e il suo stato finale è lo stato iniziale del decoder
        return logits

torch.manual_seed(7)
model = Seq2Seq(Encoder(len(src_itos), EMBEDDING_DIM, HIDDEN_SIZE),
                Decoder(len(tgt_itos), EMBEDDING_DIM, HIDDEN_SIZE)).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)       # le posizioni di padding non contano nella loss

print(model)
print(f"Parametri: {sum(p.numel() for p in model.parameters())}")

# ADDESTRAMENTO

In [ ]:
# utils.train non basta: il modello riceve due input (frase italiana e frase inglese) e produce
# una previsione per ogni parola, quindi loss e accuracy si calcolano token per token, senza il padding
def run_epoch(model, loader, optimizer=None):
    """Un'epoca di training (se c'è optimizer) o di valutazione: restituisce loss e accuracy sui token."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for src, tgt_in, tgt_out in loader:
            src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
            # TEACHER FORCING: a ogni passo il decoder riceve la parola giusta, non quella che ha predetto
            logits = model(src, tgt_in)                                          # (B, T, V)
            loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)                 # evita gradienti esplosivi nelle RNN
                optimizer.step()
            mask = tgt_out != PAD
            n = mask.sum().item()
            total_loss += loss.item() * n
            correct += ((logits.argmax(dim=-1) == tgt_out) & mask).sum().item()
            count += n
    return total_loss / count, correct / count

def train_seq2seq(model, optimizer, train_loader, test_loader, epochs):
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    for epoch in range(1, epochs + 1):
        loss, accuracy = run_epoch(model, train_loader, optimizer)
        val_loss, val_accuracy = run_epoch(model, test_loader)
        for key, value in zip(history, [loss, val_loss, accuracy, val_accuracy]):
            history[key].append(value)
        print(f"Epoch {epoch}/{epochs}: loss = {loss:.3f}, val_loss = {val_loss:.3f}, "
              f"accuracy = {accuracy:.3f}, val_accuracy = {val_accuracy:.3f}")
    return history

os.makedirs('histories', exist_ok=True)
filename = "translation_ita_eng"

hist_path = f'histories/{filename}.hist'
weights_path = f'histories/{filename}.pt'
if os.path.isfile(hist_path) and os.path.isfile(weights_path):
    with open(hist_path, 'rb') as hist_file:
        history = pickle.load(hist_file)
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
else:
    history = train_seq2seq(model, optimizer, train_loader, test_loader, EPOCHS)
    with open(hist_path, 'wb') as hist_file:
        pickle.dump(history, hist_file)
    torch.save(model.state_dict(), weights_path)

utils.hist_plot(history)

# VALUTAZIONE

In [ ]:
def translate(model, sentence, delay=0.1):
    """Traduce una frase italiana stampando una parola inglese alla volta."""
    model.eval()
    tokens = tokenize(sentence, 'it')[:MAX_LEN]
    unknown = [t for t in tokens if t not in src_stoi]
    src = torch.tensor([encode(tokens, src_stoi, MAX_LEN, pad_left=True)], device=DEVICE)
    print(sentence.lower() + (f"   (parole sconosciute: {unknown})" if unknown else ""))
    print("  -> ", end='')
    with torch.no_grad():
        h = model.encoder(src)                                     # tutta la frase italiana in un vettore
        token, prev = torch.tensor([[SOS]], device=DEVICE), None
        for _ in range(MAX_LEN + 1):
            logits, h = model.decoder(token, h)                    # un passo del decoder
            idx = logits[0, -1].argmax().item()                    # la parola più probabile

            word = tgt_itos[idx]
            print(" ", word, end='', flush=True)    # una parola alla volta
            if idx == EOS:                                         # il decoder decide da solo quando fermarsi
                break
            token = torch.tensor([[idx]], device=DEVICE)           # la parola scelta è l'input del passo successivo
            time.sleep(delay)
    print("\n")

for sentence in ["Tom è molto stanco.", "Dov'è la stazione?", "Non ho fame.", "Amo andare al parcuo!"]:
    translate(model, sentence)